In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
di = pd.read_csv("insurance.csv")

In [3]:
di.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
di.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [5]:
di.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
di.duplicated().sum()

np.int64(1)

In [7]:
di['sex'].unique()

array(['female', 'male'], dtype=object)

In [8]:
di['region'].unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [9]:
di['smoker'].unique()

array(['yes', 'no'], dtype=object)

In [10]:
x = di.drop("charges" , axis = 1)
y = di["charges"]

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train , X_test , y_train , y_test = train_test_split(x , y , test_size = 0.24 , random_state = 42)

In [14]:
cat_cols = x.select_dtypes('object').columns
num_cols = x.select_dtypes('number').columns

In [15]:
from sklearn.preprocessing import OneHotEncoder , StandardScaler

In [16]:
encoder = OneHotEncoder(drop='first', sparse_output=False , handle_unknown = 'ignore')

X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_test_cat  = encoder.transform(X_test[cat_cols])

In [17]:
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num  = scaler.transform(X_test[num_cols])

In [18]:
X_train_final = np.hstack((X_train_cat, X_train_num))

In [19]:
X_test_final  = np.hstack((X_test_cat, X_test_num))

In [22]:
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import layers, activations

In [23]:
model = Sequential([
                        Dense(50 , activation = 'relu'),
                        Dense(25 , activation = 'relu'),
                        Dense(15 , activation = 'relu'),
                        Dense(10 , activation = 'relu'),
                        Dense(5  , activation = 'relu'),
                        Dense(1  , activation = 'linear')
                ])

In [24]:
model.compile(
                optimizer = 'adam' , 
                loss      = 'mse' , 
                metrics   = ['mae']
            )

In [27]:
history = model.fit(
                        X_train_final,
                        y_train,
                        epochs     = 100,
                        batch_size = 10,
                        validation_split = 0.20 ,
                        verbose    = 1
                    )

Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 321953216.0000 - mae: 13419.1855 - val_loss: 309591168.0000 - val_mae: 12591.7598
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 313818336.0000 - mae: 13142.7793 - val_loss: 281590432.0000 - val_mae: 11594.5322
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 213890960.0000 - mae: 9686.0352 - val_loss: 121125568.0000 - val_mae: 6676.7119
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 97192136.0000 - mae: 7493.9048 - val_loss: 94942920.0000 - val_mae: 7144.2002
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 83841072.0000 - mae: 7206.5269 - val_loss: 83831168.0000 - val_mae: 6758.9438
Epoch 6/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 74430496.0000 - mae: 6804.7949 - val_loss: 74138768.0000 - val_mae: 6267.5933
Epoch 7/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 65899508.0000 - mae: 6344.7715 - val_loss: 65181144.0000 - val_mae: 5907.6499
Epoch 8/100
82/82 ━━━━━━━━━━━

In [28]:
loss , acc = model.evaluate(X_test_final , y_test , verbose = 0)

In [29]:
print(acc)
print(loss)

2444.962890625
19587722.0


In [30]:
y_pred = model.predict(X_test_final)

print(y_test[:5])
print(y_pred[:5])

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step 
764      9095.06825
887      5272.17580
890     29330.98315
1293     9301.89355
259     33750.29180
Name: charges, dtype: float64
[[ 9304.212 ]
 [ 5933.4976]
 [30714.615 ]
 [ 9655.089 ]
 [30718.074 ]]


In [35]:
from sklearn.metrics import mean_squared_error , mean_absolute_error , r2_score

y_pred = (model.predict(X_test_final))

print(f"Mean Absolute Error : {mean_absolute_error(y_test , y_pred)}")
print(f"Mean_squared_error  : {mean_squared_error(y_test , y_pred)}")
print(f"R2_Score            : {r2_score(y_test , y_pred)}")

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Mean Absolute Error : 2444.9631210514603
Mean_squared_error  : 19587723.139262453
R2_Score            : 0.8736848778923967
